In [ ]:
"""
OneDrive to Fabric Lakehouse Sync Script

This script automatically downloads assessment reports from a OneDrive/SharePoint folder
and uploads them to the appropriate folders in Microsoft Fabric Lakehouse.

Features:
- Authenticates with Microsoft Graph API
- Downloads files from OneDrive/SharePoint
- Automatically categorizes reports based on folder structure
- Uploads to Fabric Lakehouse with proper organization
- Tracks sync progress and handles errors
"""

import os
import json
import requests
from pathlib import Path
from typing import List, Dict, Optional
from datetime import datetime
import msal

# Configuration
CONFIG = {
    # Azure AD App Registration details
    'client_id': '55636aca-83c5-4e15-ab25-8df679261286',
    'tenant_id': '0b60fed4-5fc9-409d-95f2-271114f4c86f',
    
    # Microsoft Fabric details
    'workspace_id': '0f895a7e-09c6-4645-8b47-d272bc687b8a',
    'lakehouse_id': '3d0144b0-12bf-4483-9508-67b26b1fd125',  # ManagedServiceData Lakehouse
    
    # OneDrive folder path (Graph API format - relative to user's root)
    # For shared folders, you can use the sharing link directly
    'onedrive_folder_path': '/Reliance Inforcer Assessment Report',
    
    # SharePoint sharing link (Kingsley's OneDrive)
    'sharing_link': 'https://relianceinfo-my.sharepoint.com/:f:/g/personal/kingsley_relianceinfosystems_com/IgBEoymEG5W2S7cdEVa9NCK9Aa2r55L84O8nTpYQvjKi3Ek?e=GhzdcW',
    
    # Local temp directory for downloads
    'temp_dir': './temp_downloads',
    
    # Fabric folder mapping based on report type
    # These folders match the parse_assessment_pdfs.py notebook expectations
    'folder_mapping': {
        'copilot readiness': 'Files/copilot_readiness',      # Readiness assessments → copilot_readiness_* tables
        'copilot assessment': 'Files/copilot_assessment',     # Standard Copilot → copilot_assessment_* tables
        'security': 'Files/security_assessment',              # Parsed to security_assessment_* tables
        'cis': 'Files/security_assessment',                   # CIS reports go to security tables
        'm365': 'Files/security_assessment',                  # M365 reports go to security tables
        'default': 'Files/security_assessment'                # Default to security if unclear
    }
}

# Required scopes for Microsoft Graph
GRAPH_SCOPES = [
    'User.Read',
    'Files.Read.All',
    'Sites.Read.All'
]

# Required scopes for Fabric OneLake
FABRIC_SCOPES = [
    'https://storage.azure.com/.default'
]


class OneDriveToFabricSync:
    """Handles syncing files from OneDrive to Fabric Lakehouse"""
    
    def __init__(self):
        self.access_token = None  # For Graph API (OneDrive)
        self.fabric_token = None  # For OneLake (Fabric)
        self.msal_app = None  # Reuse MSAL app for both tokens
        
    def authenticate(self) -> bool:
        """
        Authenticate with Azure AD using interactive browser flow
        Returns True if successful
        """
        print("🔐 Authenticating with Microsoft...")
        
        # Create MSAL public client application (reusable)
        self.msal_app = msal.PublicClientApplication(
            CONFIG['client_id'],
            authority=f"https://login.microsoftonline.com/{CONFIG['tenant_id']}"
        )
        app = self.msal_app
        
        # Try to get token from cache first
        accounts = app.get_accounts()
        if accounts:
            print("Found existing account, attempting silent authentication...")
            result = app.acquire_token_silent(GRAPH_SCOPES, account=accounts[0])
            if result and 'access_token' in result:
                self.access_token = result['access_token']
                print("Successfully authenticated (cached)")
                return True
        
        # Use device code flow for authentication
        print("\n📱 Initiating device code authentication...\n")
        
        try:
            flow = app.initiate_device_flow(scopes=GRAPH_SCOPES)
            
            if 'user_code' not in flow:
                raise Exception("Failed to create device flow")
            
            print(flow['message'])
            print("\n⏳ Waiting for authentication (timeout: 5 minutes)...\n")
            
            # Wait for user to authenticate with timeout
            import sys
            result = app.acquire_token_by_device_flow(
                flow,
                timeout=300  # 5 minutes timeout
            )
            
        except KeyboardInterrupt:
            print("\n\n⚠️ Authentication cancelled by user")
            sys.exit(0)
        except Exception as e:
            print(f"❌ Device code authentication failed: {str(e)}")
            print("\n💡 Troubleshooting:")
            print("   - Make sure you completed the authentication in the browser")
            print("   - Check your internet connection")
            print("   - Try running the script again\n")
            return False
        
        if 'access_token' in result:
            self.access_token = result['access_token']
            print("Successfully authenticated")
            return True
        else:
            print(f"Authentication failed: {result.get('error_description', 'Unknown error')}")
            return False
    
    def get_fabric_token(self) -> bool:
        """
        Get access token for Fabric OneLake API
        Returns True if successful
        """
        if not self.msal_app:
            print("Error: MSAL app not initialized. Call authenticate() first.")
            return False
        
        print("Getting Fabric OneLake token...")
        
        # Try silent auth first
        accounts = self.msal_app.get_accounts()
        if accounts:
            result = self.msal_app.acquire_token_silent(FABRIC_SCOPES, account=accounts[0])
            if result and 'access_token' in result:
                self.fabric_token = result['access_token']
                print("Successfully obtained Fabric token (cached)")
                return True
        
        # If not in cache, use device flow
        try:
            flow = self.msal_app.initiate_device_flow(scopes=FABRIC_SCOPES)
            
            if 'user_code' not in flow:
                raise Exception("Failed to create device flow for Fabric token")
            
            print("\n** Fabric Storage Token Required **")
            print(flow['message'])
            print("\nWaiting for authentication (timeout: 5 minutes)...\n")
            
            result = self.msal_app.acquire_token_by_device_flow(
                flow,
                timeout=300
            )
            
        except KeyboardInterrupt:
            print("\n\nAuthentication cancelled by user")
            import sys
            sys.exit(0)
        except Exception as e:
            print(f"Fabric token authentication failed: {str(e)}")
            return False
        
        if 'access_token' in result:
            self.fabric_token = result['access_token']
            print("Successfully obtained Fabric token")
            return True
        else:
            print(f"Failed to get Fabric token: {result.get('error_description', 'Unknown error')}")
            return False
    
    def find_folder_by_name(self, folder_name: str, headers: dict) -> Optional[str]:
        """
        Search for a folder by name and return its ID
        """
        search_url = f'https://graph.microsoft.com/v1.0/me/drive/root/search(q=\'{folder_name}\')'
        response = requests.get(search_url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            items = data.get('value', [])
            
            for item in items:
                if 'folder' in item and folder_name.lower() in item['name'].lower():
                    print(f"✅ Found folder: {item['name']} (ID: {item['id']})")
                    return item['id']
        
        return None
    
    def list_folder_files_by_id(self, folder_id: str, headers: dict, folder_name: str = "", depth: int = 0) -> List[Dict]:
        """
        Recursively list all files in a folder by folder ID
        """
        indent = "  " * depth
        all_files = []
        
        # Build the URL using folder ID
        files_url = f'https://graph.microsoft.com/v1.0/me/drive/items/{folder_id}/children'
        
        while files_url:
            response = requests.get(files_url, headers=headers)
            
            if response.status_code != 200:
                print(f"{indent}⚠️  Could not access folder (status: {response.status_code})")
                return all_files
            
            data = response.json()
            files = data.get('value', [])
            
            print(f"{indent}   API returned {len(files)} items")
            
            file_count = 0
            folder_count = 0
            
            for item in files:
                if 'file' in item:  # It's a file
                    file_count += 1
                    print(f"{indent}   📄 File: {item['name']}")
                    all_files.append({
                        'id': item['id'],
                        'name': item['name'],
                        'size': item['size'],
                        'download_url': item.get('@microsoft.graph.downloadUrl'),
                        'path': folder_name,
                        'folder_name': folder_name
                    })
                elif 'folder' in item:  # It's a subfolder
                    folder_count += 1
                    subfolder_name = f"{folder_name}/{item['name']}" if folder_name else item['name']
                    print(f"{indent}   📁 Subfolder: {item['name']} (ID: {item['id']})")
                    subfolder_files = self.list_folder_files_by_id(item['id'], headers, subfolder_name, depth + 1)
                    all_files.extend(subfolder_files)
            
            if file_count > 0:
                print(f"{indent}✅ Found {file_count} files")
            
            # Handle pagination
            files_url = data.get('@odata.nextLink')
        
        return all_files
    
    def list_folder_files(self, folder_path: str, headers: dict, depth: int = 0) -> List[Dict]:
        """
        Recursively list all files in a folder
        """
        indent = "  " * depth
        all_files = []
        
        # Build the URL for this folder
        files_url = f'https://graph.microsoft.com/v1.0/me/drive/root:{folder_path}:/children'
        
        while files_url:
            response = requests.get(files_url, headers=headers)
            
            if response.status_code != 200:
                print(f"{indent}⚠️  Could not access path: {folder_path} (status: {response.status_code})")
                return all_files
            
            data = response.json()
            files = data.get('value', [])
            
            file_count = 0
            folder_count = 0
            
            for item in files:
                if 'file' in item:  # It's a file, not a folder
                    file_count += 1
                    all_files.append({
                        'id': item['id'],
                        'name': item['name'],
                        'size': item['size'],
                        'download_url': item.get('@microsoft.graph.downloadUrl'),
                        'path': folder_path,
                        'folder_name': folder_path.split('/')[-1]
                    })
                elif 'folder' in item:  # It's a folder, recurse into it
                    folder_count += 1
                    subfolder_path = f"{folder_path}/{item['name']}"
                    print(f"{indent}📁 Scanning subfolder: {item['name']}")
                    subfolder_files = self.list_folder_files(subfolder_path, headers, depth + 1)
                    all_files.extend(subfolder_files)
            
            if file_count > 0:
                print(f"{indent}✅ Found {file_count} files in {folder_path.split('/')[-1]}")
            
            # Handle pagination
            files_url = data.get('@odata.nextLink')
        
        return all_files
    
    def access_shared_link(self, sharing_link: str, headers: dict) -> Optional[Dict]:
        """
        Access a folder via SharePoint sharing link
        Returns dict with folder_id, drive_id, and sharing_token if successful
        """
        print(f"\n🔗 Accessing folder via sharing link...")
        
        # Encode the sharing URL
        import base64
        encoded_url = base64.b64encode(sharing_link.encode()).decode()
        # Remove padding and replace characters for URL-safe encoding
        sharing_token = encoded_url.rstrip('=').replace('/', '_').replace('+', '-')
        sharing_token = f"u!{sharing_token}"
        
        # Use the shares endpoint
        share_url = f'https://graph.microsoft.com/v1.0/shares/{sharing_token}/driveItem'
        response = requests.get(share_url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            folder_id = data.get('id')
            folder_name = data.get('name', 'Unknown')
            drive_id = data.get('parentReference', {}).get('driveId')
            
            print(f"✅ Successfully accessed shared folder: {folder_name}")
            print(f"   Drive ID: {drive_id}")
            
            return {
                'folder_id': folder_id,
                'drive_id': drive_id,
                'sharing_token': sharing_token,
                'folder_name': folder_name
            }
        else:
            print(f"❌ Could not access sharing link: {response.status_code}")
            print(f"   Response: {response.text}")
            return None
    
    def list_drive_item_files(self, drive_id: str, item_id: str, headers: dict, folder_name: str = "", depth: int = 0) -> List[Dict]:
        """
        List files from a specific drive item (for shared folders and subfolders)
        """
        indent = "  " * depth
        all_files = []
        
        # Use drives endpoint for shared items
        files_url = f'https://graph.microsoft.com/v1.0/drives/{drive_id}/items/{item_id}/children'
        
        while files_url:
            response = requests.get(files_url, headers=headers)
            
            if response.status_code != 200:
                print(f"{indent}⚠️  Could not list folder contents (status: {response.status_code})")
                return all_files
            
            data = response.json()
            files = data.get('value', [])
            
            print(f"{indent}   Found {len(files)} items")
            
            file_count = 0
            
            for item in files:
                if 'file' in item:  # It's a file
                    file_count += 1
                    print(f"{indent}   📄 {item['name']}")
                    all_files.append({
                        'id': item['id'],
                        'name': item['name'],
                        'size': item['size'],
                        'download_url': item.get('@microsoft.graph.downloadUrl'),
                        'path': folder_name,
                        'folder_name': folder_name
                    })
                elif 'folder' in item:  # It's a subfolder
                    subfolder_name = f"{folder_name}/{item['name']}" if folder_name else item['name']
                    print(f"{indent}   📁 {item['name']}/")
                    
                    # Recursively list subfolder using the same drive_id
                    subfolder_files = self.list_drive_item_files(
                        drive_id, 
                        item['id'], 
                        headers, 
                        subfolder_name, 
                        depth + 1
                    )
                    all_files.extend(subfolder_files)
            
            if file_count > 0:
                print(f"{indent}✅ {file_count} files")
            
            # Handle pagination
            files_url = data.get('@odata.nextLink')
        
        return all_files
    
    def list_shared_folder_files(self, sharing_token: str, drive_id: str, headers: dict, folder_name: str = "", depth: int = 0) -> List[Dict]:
        """
        List files from a folder accessed via sharing link
        """
        indent = "  " * depth
        all_files = []
        
        # Use shares endpoint to list children
        files_url = f'https://graph.microsoft.com/v1.0/shares/{sharing_token}/driveItem/children'
        
        while files_url:
            response = requests.get(files_url, headers=headers)
            
            if response.status_code != 200:
                print(f"{indent}⚠️  Could not list folder contents (status: {response.status_code})")
                print(f"{indent}   Response: {response.text[:200]}")
                return all_files
            
            data = response.json()
            files = data.get('value', [])
            
            print(f"{indent}   Found {len(files)} items in shared folder")
            
            file_count = 0
            
            for item in files:
                if 'file' in item:  # It's a file
                    file_count += 1
                    print(f"{indent}   📄 {item['name']}")
                    all_files.append({
                        'id': item['id'],
                        'name': item['name'],
                        'size': item['size'],
                        'download_url': item.get('@microsoft.graph.downloadUrl'),
                        'path': folder_name,
                        'folder_name': folder_name
                    })
                elif 'folder' in item:  # It's a subfolder
                    subfolder_name = f"{folder_name}/{item['name']}" if folder_name else item['name']
                    print(f"{indent}   📁 {item['name']}/")
                    
                    # For subfolders, use drive_id to access them
                    subfolder_files = self.list_drive_item_files(
                        drive_id,
                        item['id'],
                        headers,
                        subfolder_name,
                        depth + 1
                    )
                    all_files.extend(subfolder_files)
            
            if file_count > 0:
                print(f"{indent}✅ {file_count} files")
            
            # Handle pagination
            files_url = data.get('@odata.nextLink')
        
        return all_files
    
    def find_shared_folder(self, folder_name: str, headers: dict) -> Optional[str]:
        """
        Search for a folder in shared items
        """
        print(f"\n🔍 Checking shared folders...")
        shared_url = 'https://graph.microsoft.com/v1.0/me/drive/sharedWithMe'
        response = requests.get(shared_url, headers=headers)
        
        if response.status_code == 200:
            data = response.json()
            items = data.get('value', [])
            
            print(f"   Found {len(items)} shared items")
            
            for item in items:
                item_name = item.get('name', '')
                print(f"   - {item_name}")
                
                if folder_name.lower() in item_name.lower() and 'folder' in item:
                    print(f"✅ Found shared folder: {item_name} (ID: {item['id']})")
                    return item['id']
        
        return None
    
    def list_onedrive_files(self, folder_path: str = None) -> List[Dict]:
        """
        List all files in the OneDrive folder and its subfolders
        Returns list of file metadata
        """
        if not self.access_token:
            raise Exception("Not authenticated. Call authenticate() first.")
        
        folder_path = folder_path or CONFIG['onedrive_folder_path']
        folder_name = folder_path.split('/')[-1]
        
        print(f"\n📂 Looking for folder: {folder_name}")
        
        # Use Microsoft Graph API to list files
        headers = {
            'Authorization': f'Bearer {self.access_token}',
            'Content-Type': 'application/json'
        }
        
        # If a sharing link is configured, try that first
        if CONFIG.get('sharing_link'):
            share_info = self.access_shared_link(CONFIG['sharing_link'], headers)
            if share_info and share_info.get('drive_id'):
                print(f"📁 Scanning shared folder contents...")
                all_files = self.list_shared_folder_files(
                    share_info['sharing_token'],
                    share_info['drive_id'],
                    headers, 
                    share_info['folder_name']
                )
                if len(all_files) > 0:
                    print(f"\n✅ Total files found: {len(all_files)}")
                    return all_files
        
        # Try direct path
        all_files = self.list_folder_files(folder_path, headers)
        
        # If direct path didn't work, try searching for the folder
        if len(all_files) == 0:
            print(f"\n🔍 Searching for folder by name: {folder_name}")
            folder_id = self.find_folder_by_name(folder_name, headers)
            
            if folder_id:
                print(f"📁 Scanning folder contents...")
                all_files = self.list_folder_files_by_id(folder_id, headers, folder_name)
            else:
                # Try checking shared folders
                folder_id = self.find_shared_folder(folder_name, headers)
                
                if folder_id:
                    print(f"📁 Scanning shared folder contents...")
                    all_files = self.list_folder_files_by_id(folder_id, headers, folder_name)
                else:
                    print(f"\n❌ Could not find folder '{folder_name}' in your OneDrive or shared items")
                    print("💡 Make sure the folder is shared with you and you have access permissions")
        
        print(f"\n✅ Total files found: {len(all_files)}")
        return all_files
    
    def download_file(self, file_info: Dict, temp_dir: str) -> Optional[str]:
        """
        Download a file from OneDrive to local temp directory
        Returns local file path if successful
        """
        download_url = file_info.get('download_url')
        if not download_url:
            print(f"⚠️  No download URL for {file_info['name']}")
            return None
        
        # Create temp directory if it doesn't exist
        os.makedirs(temp_dir, exist_ok=True)
        
        local_path = os.path.join(temp_dir, file_info['name'])
        
        print(f"⬇️  Downloading: {file_info['name']} ({file_info['size']} bytes)")
        
        # Download file with retry logic
        max_retries = 3
        for attempt in range(max_retries):
            try:
                response = requests.get(download_url, timeout=30)
                
                if response.status_code == 200:
                    with open(local_path, 'wb') as f:
                        f.write(response.content)
                    print(f"   ✅ Downloaded to: {local_path}")
                    return local_path
                else:
                    print(f"   ❌ Download failed: {response.status_code}")
                    if attempt < max_retries - 1:
                        print(f"   Retrying ({attempt + 2}/{max_retries})...")
                        continue
                    return None
            except Exception as e:
                print(f"   ❌ Download error: {str(e)}")
                if attempt < max_retries - 1:
                    print(f"   Retrying ({attempt + 2}/{max_retries})...")
                    import time
                    time.sleep(2)
                    continue
                return None
    
    def categorize_file(self, file_name: str, file_path: str = '') -> tuple:
        """
        Determine the appropriate Fabric folder based on file name and path
        Returns tuple of (folder_path, report_type)
        
        Report types:
        - 'copilot_readiness': Copilot Readiness Assessment (with gap analysis)
        - 'copilot_assessment': Standard Copilot Assessment/Summary
        - 'security': Security Assessment (CIS, M365, etc.)
        """
        file_name_lower = file_name.lower()
        path_lower = file_path.lower()
        
        # Priority 1: Check for Copilot Assessment/Summary FIRST (more common)
        # Matches: "Summary Copilot Assessment", "Copilot Assessment Summary", etc.
        copilot_assessment_patterns = [
            'summary copilot assessment',
            'copilot assessment summary',
            'summary copilot',
            'copilot summary',
            'copilot assessment',
            'co-pilot assessment'
        ]
        if any(pattern in file_name_lower for pattern in copilot_assessment_patterns) or \
           ('copilot' in file_name_lower and 'summary' in file_name_lower):
            return (CONFIG['folder_mapping']['copilot assessment'], 'COPILOT_ASSESSMENT')
        
        # Priority 2: Check for Copilot Readiness (more specific - includes "readiness")
        # Matches: "Copilot Readiness Assessment", "Readiness Assessment", etc.
        if ('copilot' in file_name_lower and 'readiness' in file_name_lower) or \
           ('readiness' in file_name_lower and 'assessment' in file_name_lower):
            return (CONFIG['folder_mapping']['copilot readiness'], 'COPILOT_READINESS')
        
        # Priority 3: Generic copilot (fallback to copilot assessment)
        if 'copilot' in file_name_lower or 'co-pilot' in file_name_lower:
            return (CONFIG['folder_mapping']['copilot assessment'], 'COPILOT_ASSESSMENT')
        
        # Priority 4: Security-related keywords
        security_keywords = ['security', 'cis', 'm365', 'microsoft 365', 'compliance', 'benchmark']
        if any(keyword in file_name_lower for keyword in security_keywords) or \
           any(keyword in path_lower for keyword in security_keywords):
            return (CONFIG['folder_mapping']['security'], 'SECURITY')
        
        # Default to security assessment
        print(f"   ⚠️  Unable to categorize '{file_name}' - defaulting to security assessment")
        return (CONFIG['folder_mapping']['default'], 'SECURITY')
    
    def upload_to_fabric(self, local_file_path: str, fabric_folder: str, file_name: str, upload_date: str = None) -> str:
        """
        Upload a file to Fabric Lakehouse using Azure Data Lake Storage Gen2 API
        Returns "success", "skipped", or "failed"
        
        Args:
            local_file_path: Path to local file
            fabric_folder: Base folder in Fabric (e.g., Files/copilot_readiness)
            file_name: Name of the file to upload
            upload_date: Optional date folder (e.g., '2026-06-13'). If provided, creates subfolder.
        """
        # Ensure we have Fabric token
        if not self.fabric_token:
            if not self.get_fabric_token():
                print("   Failed to obtain Fabric token")
                return "failed"
        
        workspace_id = CONFIG['workspace_id']
        lakehouse_id = CONFIG['lakehouse_id']
        
        # OneLake uses Azure Data Lake Storage Gen2 API
        # Add date-based subfolder if provided
        if upload_date:
            fabric_path = f"{fabric_folder}/{upload_date}/{file_name}"
        else:
            fabric_path = f"{fabric_folder}/{file_name}"
        
        # OneLake endpoint format
        base_url = (
            f"https://onelake.dfs.fabric.microsoft.com/"
            f"{workspace_id}/{lakehouse_id}/{fabric_path}"
        )
        
        # Check if file already exists (for resume capability)
        headers_check = {
            'Authorization': f'Bearer {self.fabric_token}',
            'x-ms-version': '2023-01-03'
        }
        
        check_response = requests.head(base_url, headers=headers_check)
        if check_response.status_code == 200:
            print(f"⏭️  Skipping (already uploaded): {fabric_path}")
            return "skipped"
        
        print(f"Uploading to Fabric: {fabric_path}")
        
        # Read file content
        with open(local_file_path, 'rb') as f:
            file_content = f.read()
        
        file_size = len(file_content)
        
        # Step 1: Create the file (PUT with resource=file)
        headers = {
            'Authorization': f'Bearer {self.fabric_token}',
            'x-ms-version': '2023-01-03',
            'Content-Length': '0'
        }
        
        create_response = requests.put(
            base_url,
            headers=headers,
            params={'resource': 'file'}
        )
        
        if create_response.status_code not in [200, 201]:
            print(f"   Failed to create file: {create_response.status_code}")
            print(f"   Response: {create_response.text}")
            return "failed"
        
        # Step 2: Append data (PATCH with action=append&position=0)
        headers = {
            'Authorization': f'Bearer {self.fabric_token}',
            'x-ms-version': '2023-01-03',
            'Content-Type': 'application/octet-stream',
            'Content-Length': str(file_size)
        }
        
        append_response = requests.patch(
            base_url,
            headers=headers,
            params={'action': 'append', 'position': '0'},
            data=file_content
        )
        
        if append_response.status_code not in [200, 202]:
            print(f"   Failed to append data: {append_response.status_code}")
            print(f"   Response: {append_response.text}")
            return "failed"
        
        # Step 3: Flush to finalize (PATCH with action=flush&position=file_size)
        headers = {
            'Authorization': f'Bearer {self.fabric_token}',
            'x-ms-version': '2023-01-03',
            'Content-Length': '0'
        }
        
        flush_response = requests.patch(
            base_url,
            headers=headers,
            params={'action': 'flush', 'position': str(file_size)}
        )
        
        if flush_response.status_code in [200, 201]:
            print(f"   Upload successful")
            return "success"
        else:
            print(f"   Failed to flush: {flush_response.status_code}")
            print(f"   Response: {flush_response.text}")
            return "failed"
    
    def sync_all(self, dry_run: bool = False, use_date_folders: bool = True) -> Dict:
        """
        Sync all files from OneDrive to Fabric
        
        Args:
            dry_run: If True, only shows what would be synced without uploading
            use_date_folders: If True, organizes uploads into date-based subfolders (YYYY-MM-DD)
        
        Returns:
            Dictionary with sync statistics
        """
        from datetime import datetime
        
        stats = {
            'total_files': 0,
            'downloaded': 0,
            'uploaded': 0,
            'failed': 0,
            'skipped': 0,
            'copilot_reports': 0,
            'security_reports': 0
        }
        
        # Generate upload date folder (YYYY-MM-DD format)
        upload_date = datetime.now().strftime('%Y-%m-%d') if use_date_folders else None
        
        print("\n" + "="*60)
        print("🔄 Starting OneDrive to Fabric Sync")
        print("="*60)
        print(f"📍 Target Lakehouse: {CONFIG['lakehouse_id']}")
        print(f"📁 OneDrive Path: {CONFIG['onedrive_folder_path']}")
        if upload_date:
            print(f"📅 Upload Date Folder: {upload_date}")
        
        # Authenticate
        if not self.authenticate():
            print("\n❌ Sync aborted: Authentication failed")
            return stats
        
        # List files
        files = self.list_onedrive_files()
        stats['total_files'] = len(files)
        
        if not files:
            print("\n⚠️  No files found to sync")
            return stats
        
        # Create temp directory
        temp_dir = CONFIG['temp_dir']
        os.makedirs(temp_dir, exist_ok=True)
        
        # Process each file
        print(f"\n📦 Processing {len(files)} files...")
        print("-" * 60)
        
        for i, file_info in enumerate(files, 1):
            print(f"\n[{i}/{len(files)}] {file_info['name']}")
            
            # Only process PDF files
            if not file_info['name'].lower().endswith('.pdf'):
                print(f"   ⏭️  Skipping non-PDF file")
                stats['skipped'] += 1
                continue
            
            # Download file
            local_path = self.download_file(file_info, temp_dir)
            
            if not local_path:
                stats['failed'] += 1
                continue
            
            stats['downloaded'] += 1
            
            # Determine target folder and report type
            fabric_folder, report_type = self.categorize_file(
                file_info['name'],
                file_info.get('path', '')
            )
            
            print(f"   📂 Target folder: {fabric_folder}")
            print(f"   🏷️  Report type: {report_type.upper()}")
            
            if dry_run:
                target_path = f"{fabric_folder}/{upload_date}" if upload_date else fabric_folder
                print(f"   🔍 DRY RUN - Would upload to: {target_path}")
                stats['uploaded'] += 1
                if report_type == 'copilot':
                    stats['copilot_reports'] += 1
                else:
                    stats['security_reports'] += 1
            else:
                # Upload to Fabric with date-based subfolder
                result = self.upload_to_fabric(local_path, fabric_folder, file_info['name'], upload_date)
                
                if result == "success":
                    stats['uploaded'] += 1
                    if report_type == 'copilot':
                        stats['copilot_reports'] += 1
                    else:
                        stats['security_reports'] += 1
                elif result == "skipped":
                    stats['skipped'] += 1
                else:  # "failed" or other error
                    stats['failed'] += 1
            
            # Clean up local file
            try:
                os.remove(local_path)
            except:
                pass
        
        # Clean up temp directory
        try:
            os.rmdir(temp_dir)
        except:
            pass
        
        # Print summary
        print("\n" + "="*60)
        print("📊 Sync Summary")
        print("="*60)
        print(f"Total files found:  {stats['total_files']}")
        print(f"Downloaded:         {stats['downloaded']}")
        print(f"Uploaded:           {stats['uploaded']}")
        print(f"  • Copilot:        {stats['copilot_reports']}")
        print(f"  • Security:       {stats['security_reports']}")
        print(f"Failed:             {stats['failed']}")
        print(f"Skipped:            {stats['skipped']}")
        print("="*60)
        
        # Show next steps if files were uploaded
        if stats['uploaded'] > 0 and not dry_run:
            print("\n✅ Files uploaded successfully!")
            print("\n📊 Next Steps:")
            print("1. Open Microsoft Fabric workspace")
            print("2. Navigate to your Lakehouse: ManagedServiceData")
            print("3. Run the 'parse_assessment_pdfs' notebook to process the files")
            print("4. Check the following tables:")
            if stats['copilot_reports'] > 0:
                print("   • copilot_readiness_assessments")
                print("   • copilot_readiness_categories")
                print("   • copilot_readiness_checks")
            if stats['security_reports'] > 0:
                print("   • security_assessment_assessments")
                print("   • security_assessment_categories")
                print("   • security_assessment_checks")
        
        return stats


def main():
    """Main entry point for the sync script"""
    import argparse
    
    parser = argparse.ArgumentParser(
        description='Sync assessment reports from OneDrive to Fabric Lakehouse'
    )
    parser.add_argument(
        '--dry-run',
        action='store_true',
        help='Show what would be synced without actually uploading'
    )
    parser.add_argument(
        '--lakehouse-id',
        help='Fabric Lakehouse ID (overrides config)'
    )
    parser.add_argument(
        '--no-date-folders',
        action='store_true',
        help='Disable date-based subfolders (uploads directly to base folders)'
    )
    
    args = parser.parse_args()
    
    # Update config if lakehouse ID provided
    if args.lakehouse_id:
        CONFIG['lakehouse_id'] = args.lakehouse_id
        print(f"Using Lakehouse ID from command line: {args.lakehouse_id}")
    
    # Create syncer and run
    syncer = OneDriveToFabricSync()
    stats = syncer.sync_all(dry_run=args.dry_run, use_date_folders=not args.no_date_folders)
    
    # Exit code based on results
    if stats['failed'] > 0:
        exit(1)
    else:
        exit(0)


if __name__ == '__main__':
    main()


# Dynamics 365 CRM Integration

This section ingests account data from Dynamics 365 CRM into the Fabric Lakehouse.

**Schema:**
- Name
- Relationship Type
- Owner
- Territory  
- Status

In [ ]:
# Install Dataverse SDK for Python
%pip install PowerPlatform-Dataverse-client azure-identity --quiet

In [ ]:
from azure.identity import InteractiveBrowserCredential, DefaultAzureCredential
from PowerPlatform.Dataverse.client import DataverseClient
from PowerPlatform.Dataverse.models.filters import col
import pandas as pd
from datetime import datetime

# Dynamics 365 CRM Configuration - Reliance Nigeria
# Nigeria environment URL (Europe region)
# Format: https://{your-org}.crm4.dynamics.com (Europe/Nigeria region)
CRM_BASE_URL = "https://relianceinfo.crm4.dynamics.com"  # Reliance Nigeria CRM

# Lakehouse configuration (using existing lakehouse from CONFIG above)
TARGET_TABLE_NAME = "crm_accounts"
LAKEHOUSE_PATH = f"Tables/{TARGET_TABLE_NAME}"

print(f"🔗 Connecting to Dynamics 365 CRM: {CRM_BASE_URL}")

In [ ]:
# Authenticate and create Dataverse client
# Using InteractiveBrowserCredential for user authentication
# For scheduled/automated runs, consider using DefaultAzureCredential with managed identity

try:
    print("🔐 Authenticating...")
    # Try DefaultAzureCredential first (works with managed identity in scheduled runs)
    credential = DefaultAzureCredential()
    crm_client = DataverseClient(base_url=CRM_BASE_URL, credential=credential)
    
    # Test connection by querying account metadata
    print("✅ Successfully connected to Dynamics 365 CRM")
    
except Exception as e:
    print(f"⚠️ DefaultAzureCredential failed, falling back to interactive auth: {e}")
    # Fallback to interactive browser authentication for manual runs
    credential = InteractiveBrowserCredential()
    crm_client = DataverseClient(base_url=CRM_BASE_URL, credential=credential)
    print("✅ Successfully connected to Dynamics 365 CRM (interactive auth)")

In [ ]:
# Query CRM Accounts with the required fields
print("📊 Querying CRM accounts...")

# Build query for active accounts with required fields
# Note: Field names in Dynamics 365:
# - name: Account name
# - customertypecode: Customer/Relationship type (picklist)
# - ownerid: Owner (lookup to systemuser)
# - territoryid: Territory (lookup to territory table)
# - statecode: Status (Active=0, Inactive=1)
# - statuscode: Detailed status code

try:
    # Query accounts with expanded lookups to get owner and territory names
    accounts_df = (
        crm_client.query.builder("account")
        .select(
            "accountid",
            "name",
            "customertypecode",           # Relationship type
            "statecode",                  # Status (0=Active, 1=Inactive)
            "statuscode",                 # Detailed status
            "_ownerid_value",             # Owner ID
            "_territoryid_value"          # Territory ID (if used)
        )
        .where(col("statecode") == 0)     # Filter for active accounts only
        .top(10000)                       # Limit to prevent timeout, adjust as needed
        .execute()
        .to_dataframe()
    )
    
    print(f"✅ Retrieved {len(accounts_df)} active accounts from CRM")
    
    # Display sample
    if len(accounts_df) > 0:
        print("\n📋 Sample of raw data:")
        print(accounts_df.head(3))
    
except Exception as e:
    print(f"❌ Error querying accounts: {e}")
    raise

In [ ]:
# Enrich data with Owner and Territory names
print("\n🔍 Enriching data with owner and territory information...")

# Get unique owner IDs
if '_ownerid_value' in accounts_df.columns:
    unique_owner_ids = accounts_df['_ownerid_value'].dropna().unique()
    
    if len(unique_owner_ids) > 0:
        print(f"   Fetching {len(unique_owner_ids)} unique owners...")
        
        # Query systemuser table for owner names
        owners_df = (
            crm_client.query.builder("systemuser")
            .select("systemuserid", "fullname")
            .where(col("systemuserid").is_in(list(unique_owner_ids)))
            .execute()
            .to_dataframe()
        )
        
        # Create lookup dictionary
        owner_lookup = dict(zip(owners_df['systemuserid'], owners_df['fullname']))
        
        # Map owner names
        accounts_df['Owner'] = accounts_df['_ownerid_value'].map(owner_lookup).fillna("Unassigned")
        print(f"   ✅ Mapped {len(owner_lookup)} owner names")
    else:
        accounts_df['Owner'] = "Unassigned"
else:
    accounts_df['Owner'] = "Unassigned"

# Get territory names (if territoryid is used in your org)
if '_territoryid_value' in accounts_df.columns:
    unique_territory_ids = accounts_df['_territoryid_value'].dropna().unique()
    
    if len(unique_territory_ids) > 0:
        print(f"   Fetching {len(unique_territory_ids)} unique territories...")
        try:
            territories_df = (
                crm_client.query.builder("territory")
                .select("territoryid", "name")
                .where(col("territoryid").is_in(list(unique_territory_ids)))
                .execute()
                .to_dataframe()
            )
            
            territory_lookup = dict(zip(territories_df['territoryid'], territories_df['name']))
            accounts_df['Territory'] = accounts_df['_territoryid_value'].map(territory_lookup).fillna("Not Assigned")
            print(f"   ✅ Mapped {len(territory_lookup)} territory names")
        except Exception as e:
            print(f"   ⚠️ Could not fetch territories (table may not exist): {e}")
            accounts_df['Territory'] = "Not Assigned"
    else:
        accounts_df['Territory'] = "Not Assigned"
else:
    accounts_df['Territory'] = "Not Assigned"

print("✅ Data enrichment complete")

In [ ]:
# Transform to final schema
print("\n🔄 Transforming to target schema...")

# Map customertypecode picklist values to readable labels
# Common Dynamics 365 customer type codes:
# 1 = Competitor, 2 = Consultant, 3 = Customer, 4 = Investor, 5 = Partner, 
# 6 = Influencer, 7 = Press, 8 = Prospect, 9 = Reseller, 10 = Supplier, 11 = Vendor, 12 = Other
customer_type_mapping = {
    1: "Competitor",
    2: "Consultant", 
    3: "Customer",
    4: "Investor",
    5: "Partner",
    6: "Influencer",
    7: "Press",
    8: "Prospect",
    9: "Reseller",
    10: "Supplier",
    11: "Vendor",
    12: "Other"
}

# Map status codes to readable labels
status_mapping = {
    0: "Active",
    1: "Inactive"
}

# Create final dataframe with desired schema
crm_accounts_final = pd.DataFrame({
    'AccountID': accounts_df['accountid'],
    'Name': accounts_df['name'],
    'RelationshipType': accounts_df['customertypecode'].map(customer_type_mapping).fillna("Unknown"),
    'Owner': accounts_df['Owner'],
    'Territory': accounts_df['Territory'],
    'Status': accounts_df['statecode'].map(status_mapping).fillna("Unknown"),
    'LastSyncDate': datetime.now().strftime("%Y-%m-%d %H:%M:%S")
})

print(f"✅ Transformed {len(crm_accounts_final)} records")
print(f"\n📊 Final schema:")
print(crm_accounts_final.info())
print(f"\n📋 Sample data:")
print(crm_accounts_final.head(10))

In [ ]:
# Save to Fabric Lakehouse
print(f"\n💾 Saving to Fabric Lakehouse table '{TARGET_TABLE_NAME}'...")

try:
    # Convert to Spark DataFrame
    spark_df = spark.createDataFrame(crm_accounts_final)
    
    # Write to Delta table in the lakehouse
    # Mode 'overwrite' replaces the entire table on each run
    # Use 'append' to add new records, or implement upsert logic for updates
    spark_df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(TARGET_TABLE_NAME)
    
    print(f"✅ Successfully saved {len(crm_accounts_final)} CRM accounts to lakehouse table '{TARGET_TABLE_NAME}'")
    print(f"   Table location: {LAKEHOUSE_PATH}")
    
    # Verify the save
    verification_df = spark.sql(f"SELECT COUNT(*) as count FROM {TARGET_TABLE_NAME}")
    record_count = verification_df.collect()[0]['count']
    print(f"✅ Verification: Table now contains {record_count} records")
    
except Exception as e:
    print(f"❌ Error saving to lakehouse: {e}")
    raise

print("\n🎉 CRM account ingestion complete!")

In [ ]:
# Optional: View summary statistics and data quality
print("\n📈 Data Summary:")
print("=" * 60)

# Summary by relationship type
print("\n📊 Accounts by Relationship Type:")
relationship_summary = crm_accounts_final['RelationshipType'].value_counts()
print(relationship_summary)

# Summary by owner
print("\n👤 Accounts by Owner (Top 10):")
owner_summary = crm_accounts_final['Owner'].value_counts().head(10)
print(owner_summary)

# Summary by territory
print("\n🗺️ Accounts by Territory:")
territory_summary = crm_accounts_final['Territory'].value_counts()
print(territory_summary)

# Summary by status
print("\n📍 Accounts by Status:")
status_summary = crm_accounts_final['Status'].value_counts()
print(status_summary)

print("\n" + "=" * 60)
print("✅ All statistics generated successfully")